# Breaking Bad — Decompress Artifact Subset (10/object) trên Kaggle

**Mục tiêu:** decompress 10 fracture ngẫu nhiên/object (5 mode + 5 fractured) cho subset `artifact`, output sẵn sàng dùng cho training DiffusionNet.

**Chiến lược**: sub-sample TRƯỚC khi decompress để tránh vượt giới hạn 20GB của Kaggle output.

**Kết quả**: folder `artifact/` chứa mesh `.obj` thật, mỗi object có ~10 fractures, mỗi fracture có nhiều `piece_X.obj`. Tổng ~3–5 GB.

## Yêu cầu trước khi chạy

1. **Upload file lên Kaggle Dataset** (Kaggle tự động giải nén `.zip` khi upload — không cần unzip thủ công):
   - `artifact_compressed.zip` HOẶC `volume_constrained-artifact_compressed.zip` (chọn 1, đừng up cả 2 vì cùng tên folder gốc → conflict)
   - `data_split.tar.gz`

2. **Notebook settings (panel phải)**:
   - Accelerator: CPU (decompress không cần GPU)
   - Internet: **ON**
   - Persistence: **Files only**
   - Attach dataset qua **Add Input**

3. Sau khi notebook chạy xong, dùng **Save Version** để snapshot output rồi tạo Kaggle Dataset mới từ `/kaggle/working/artifact/` để các notebook training sau dùng được.


## 1. Cấu hình + auto-detect path

Cell này tự tìm vị trí của `artifact_compressed/` và `data_split/` trong `/kaggle/input/`, bất kể slug dataset là gì.

In [ ]:
import os, sys, shutil, random, json
from pathlib import Path
from collections import deque

# === CONFIG ===
SEED = 42
N_MODES_PER_OBJECT = 5      # số mode_X giữ lại
N_FRACTURED_PER_OBJECT = 5  # số fractured_X giữ lại

# === PATHS ===
WORK_DIR = '/kaggle/working'
FILTERED_ROOT = f'{WORK_DIR}/bb_filtered'   # compressed đã sub-sample
OUTPUT_ROOT = f'{WORK_DIR}/artifact'        # output cuối — meshes thật

random.seed(SEED)

# === AUTO-DETECT INPUT PATHS (BFS với depth limit, không dùng rglob) ===
INPUT_BASE = Path('/kaggle/input')

print('Folders trong /kaggle/input/:')
for d in INPUT_BASE.iterdir():
    print(f'  - {d.name}')
print()

def find_artifact_root(base: Path, max_depth: int = 5):
    """Tìm folder chứa nhiều thư mục con kết thúc bằng `_sf` (đây là root của artifact)."""
    best = None
    best_count = 0
    queue = deque([(base, 0)])
    while queue:
        path, depth = queue.popleft()
        if depth > max_depth:
            continue
        try:
            children = list(path.iterdir())
        except (PermissionError, OSError):
            continue
        sf_count = sum(1 for c in children if c.is_dir() and c.name.endswith('_sf'))
        if sf_count > best_count:
            best_count = sf_count
            best = path
        for c in children:
            if c.is_dir() and not c.name.endswith('_sf'):
                queue.append((c, depth + 1))
    return best if best_count >= 50 else None

def find_named_folder(base: Path, name: str, max_depth: int = 5):
    """Tìm folder có tên cụ thể bằng BFS."""
    queue = deque([(base, 0)])
    while queue:
        path, depth = queue.popleft()
        if depth > max_depth:
            continue
        try:
            children = list(path.iterdir())
        except (PermissionError, OSError):
            continue
        for c in children:
            if c.is_dir():
                if c.name == name:
                    return c
                queue.append((c, depth + 1))
    return None

def find_file_named(base: Path, name: str, max_depth: int = 5):
    queue = deque([(base, 0)])
    while queue:
        path, depth = queue.popleft()
        if depth > max_depth:
            continue
        try:
            children = list(path.iterdir())
        except (PermissionError, OSError):
            continue
        for c in children:
            if c.is_file() and c.name == name:
                return c
            if c.is_dir():
                queue.append((c, depth + 1))
    return None

# Tìm root chứa các object _sf
ARTIFACT_SRC = find_artifact_root(INPUT_BASE)

if ARTIFACT_SRC is None:
    raise FileNotFoundError(
        '❌ Không tìm thấy folder chứa các object *_sf/ trong /kaggle/input. '
        'Kiểm tra dataset đã attach chưa.'
    )

n_objects = sum(1 for c in ARTIFACT_SRC.iterdir() if c.is_dir() and c.name.endswith('_sf'))
print(f'✅ ARTIFACT_SRC = {ARTIFACT_SRC}')
print(f'   Số object (*_sf folders): {n_objects}')

# Tìm data_split — có thể là folder hoặc tar.gz
DATA_SPLIT_SRC = find_named_folder(INPUT_BASE, 'data_split')
if DATA_SPLIT_SRC is not None:
    print(f'✅ DATA_SPLIT_SRC = {DATA_SPLIT_SRC} (đã giải nén)')
else:
    tar_file = find_file_named(INPUT_BASE, 'data_split.tar.gz')
    if tar_file is not None:
        DATA_SPLIT_SRC = tar_file
        print(f'⚠️ Tìm thấy {DATA_SPLIT_SRC} (chưa giải nén — sẽ extract ở cell tiếp)')
    else:
        print('⚠️ Không tìm thấy data_split (sẽ skip)')

print(f'\nSẽ giữ {N_MODES_PER_OBJECT} mode + {N_FRACTURED_PER_OBJECT} fractured per object')

## 2. Sub-sample: chỉ giữ 10 fracture/object

Với mỗi object, random pick:
- 5 `mode_X` (từ 20 mode có sẵn)
- 5 `fractured_X` (từ 80 fractured có sẵn)

Copy gọn vào `bb_filtered/` để chạy decompress. Vì `/kaggle/input` là **read-only**, ta copy sang `/kaggle/working`.

In [ ]:
src = ARTIFACT_SRC
dst = Path(FILTERED_ROOT) / 'artifact_compressed'
dst.mkdir(parents=True, exist_ok=True)

objects = sorted([d for d in src.iterdir() if d.is_dir()])
print(f'Tổng số object: {len(objects)}')

kept_summary = []

for i, obj_dir in enumerate(objects):
    obj_dst = dst / obj_dir.name
    obj_dst.mkdir(exist_ok=True)
    
    # Copy 2 file gốc bắt buộc
    for required in ['compressed_mesh.obj', 'compressed_data.npz']:
        src_file = obj_dir / required
        if src_file.exists():
            shutil.copy(src_file, obj_dst / required)
    
    # Liệt kê các mode_X và fractured_X
    modes = sorted([d for d in obj_dir.iterdir() if d.is_dir() and d.name.startswith('mode_')])
    fractured = sorted([d for d in obj_dir.iterdir() if d.is_dir() and d.name.startswith('fractured_')])
    
    # Pick random
    picked_modes = random.sample(modes, min(N_MODES_PER_OBJECT, len(modes)))
    picked_fractured = random.sample(fractured, min(N_FRACTURED_PER_OBJECT, len(fractured)))
    
    # Copy folder
    for d in picked_modes + picked_fractured:
        shutil.copytree(d, obj_dst / d.name, dirs_exist_ok=True)
    
    kept_summary.append({
        'object': obj_dir.name,
        'n_modes_total': len(modes),
        'n_fractured_total': len(fractured),
        'kept_modes': [m.name for m in picked_modes],
        'kept_fractured': [f.name for f in picked_fractured],
    })
    
    if (i + 1) % 50 == 0:
        print(f'  Progress: {i+1}/{len(objects)} object đã sub-sample')

# Save manifest
with open(f'{WORK_DIR}/sample_manifest.json', 'w') as f:
    json.dump({'seed': SEED, 'objects': kept_summary}, f, indent=2)

print(f'\n✅ Đã sub-sample {len(kept_summary)} object.')
print(f'Manifest lưu ở: {WORK_DIR}/sample_manifest.json')

# Copy data_split sang filtered root (decompress script cần)
data_split_dst = Path(FILTERED_ROOT) / 'data_split'
if DATA_SPLIT_SRC is not None:
    if DATA_SPLIT_SRC.is_dir():
        # đã giải nén — copy folder
        shutil.copytree(DATA_SPLIT_SRC, data_split_dst, dirs_exist_ok=True)
    else:
        # vẫn là tar.gz — extract
        os.makedirs(FILTERED_ROOT, exist_ok=True)
        !tar -xzf '{DATA_SPLIT_SRC}' -C '{FILTERED_ROOT}/'
    print(f'✅ data_split copied to {data_split_dst}')

print()
print('Disk usage:')
!du -sh {FILTERED_ROOT}/*

## 3. Clone repo decompress + cài dependencies

In [ ]:
REPO_DIR = f'{WORK_DIR}/bb-decompress'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Breaking-Bad-Dataset/Breaking-Bad-Dataset.github.io.git {REPO_DIR}

print('Files trong repo:')
!ls {REPO_DIR}/

In [ ]:
# Cài dependencies cho decompress
# QUAN TRỌNG: gpytoolbox==0.2.0 KHÔNG có wheel cho Python 3.10+ → build from source fail.
# Giải pháp: cài bản mới hơn (>=0.3.0) có pre-built wheel. API tương thích ngược cho các function decompress.py dùng.

!pip install -q numpy scipy tqdm

# libigl: dùng --only-binary để chắc chắn lấy wheel, không build
!pip install -q --only-binary :all: libigl

# gpytoolbox: thử bản mới nhất trước (có wheel cho Python 3.10/3.11)
# Nếu fail, fallback: thử bản 0.3.0 / 0.3.3 / 0.4.x
import subprocess

def try_install_gpytoolbox():
    candidates = [
        ['pip', 'install', '-q', '--only-binary', ':all:', 'gpytoolbox'],
        ['pip', 'install', '-q', '--only-binary', ':all:', 'gpytoolbox==0.3.3'],
        ['pip', 'install', '-q', '--only-binary', ':all:', 'gpytoolbox==0.3.0'],
    ]
    for cmd in candidates:
        print(f'Thử: {" ".join(cmd)}')
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            # Test import
            test = subprocess.run(['python', '-c', 'import gpytoolbox; print(gpytoolbox.__version__)'],
                                  capture_output=True, text=True)
            if test.returncode == 0:
                print(f'✅ Cài thành công: gpytoolbox {test.stdout.strip()}')
                return True
        else:
            print(f'  Không được, thử bản khác...')
    return False

if not try_install_gpytoolbox():
    raise RuntimeError('❌ Không cài được gpytoolbox. Cần xử lý thủ công.')

# Verify import
import igl
import gpytoolbox
print(f'\nlibigl: {igl.__version__ if hasattr(igl, "__version__") else "installed"}')
print(f'gpytoolbox: {gpytoolbox.__version__ if hasattr(gpytoolbox, "__version__") else "installed"}')

# ============================================================
# PATCH decompress.py: libigl mới đã xóa `resolve_duplicated_faces`
# Thêm fallback function bằng numpy để decompress.py vẫn chạy được.
# ============================================================
import os

script_path = f'{REPO_DIR}/decompress.py'

if not os.path.exists(script_path):
    print(f'⚠️ Không thấy {script_path}, clone lại repo...')
    import shutil as _sh
    if os.path.exists(REPO_DIR):
        _sh.rmtree(REPO_DIR)
    os.system(f'git clone https://github.com/Breaking-Bad-Dataset/Breaking-Bad-Dataset.github.io.git {REPO_DIR}')

if not os.path.exists(script_path):
    raise FileNotFoundError(f'❌ {script_path} không tồn tại. Cần debug thủ công.')

with open(script_path, 'r') as f:
    content = f.read()

if 'COMPAT_PATCH' not in content:
    PATCH = '''
# === COMPAT_PATCH ===
import numpy as _np_compat
if not hasattr(igl, 'resolve_duplicated_faces'):
    def _resolve_duplicated_faces_compat(F):
        F = _np_compat.asarray(F)
        Fs = _np_compat.sort(F, axis=1)
        _, uidx = _np_compat.unique(Fs, axis=0, return_index=True)
        uidx = _np_compat.sort(uidx)
        return F[uidx], uidx
    igl.resolve_duplicated_faces = _resolve_duplicated_faces_compat
# === END COMPAT_PATCH ===
'''
    content = content.replace('import igl', 'import igl' + PATCH, 1)
    with open(script_path, 'w') as f:
        f.write(content)
    print('\n✅ Đã patch decompress.py (libigl compat)')
else:
    print('\nℹ️  decompress.py đã được patch trước đó, bỏ qua.')

## 4. Chạy decompress

Script `decompress.py` sẽ đọc compressed format và xuất ra `piece_X.obj` cho mỗi mảnh vỡ.

In [ ]:
import subprocess

# === PATCH decompress.py: thêm function resolve_duplicated_faces đã bị xóa trong libigl mới ===
patch_code = '''# === Patch: polyfill cho resolve_duplicated_faces bị xóa trong libigl mới ===
import igl as _bb_igl
import numpy as _bb_np

if not hasattr(_bb_igl, 'resolve_duplicated_faces'):
    def _bb_resolve_duplicated_faces(F):
        """Tìm các face unique. F có shape (N, 3) chỉ số vertex."""
        F = _bb_np.asarray(F)
        # Sort mỗi row để so sánh canonical
        F_sorted = _bb_np.sort(F, axis=1)
        # Lấy unique theo row
        _, unique_idx, inverse = _bb_np.unique(F_sorted, axis=0, return_index=True, return_inverse=True)
        return F[unique_idx], inverse
    _bb_igl.resolve_duplicated_faces = _bb_resolve_duplicated_faces
# === Hết patch ===

'''

decompress_path = f'{REPO_DIR}/decompress.py'
with open(decompress_path, 'r') as f:
    original = f.read()

# Chỉ patch 1 lần
if '_bb_resolve_duplicated_faces' not in original:
    with open(decompress_path, 'w') as f:
        f.write(patch_code + original)
    print('✅ Đã patch decompress.py')
else:
    print('⚠️ decompress.py đã được patch trước đó')

# === Chạy decompress ===
cmd = [
    'python', decompress_path,
    '--data_root', FILTERED_ROOT,
    '--subset', 'artifact',
]

print(f'\nĐang chạy: {" ".join(cmd)}')
print('(Có thể mất 20-40 phút cho ~2000 fractures...)')
print()

result = subprocess.run(cmd, capture_output=False, cwd=REPO_DIR)
print(f'\nReturn code: {result.returncode}')

## 5. Kiểm tra output

In [ ]:
# Output mong đợi: {FILTERED_ROOT}/artifact/{object_id}_sf/{mode|fractured}_X/piece_X.obj
decompressed_root = Path(FILTERED_ROOT) / 'artifact'

if not decompressed_root.exists():
    print(f'❌ Không thấy {decompressed_root}. Kiểm tra log decompress.')
else:
    objects = sorted([d for d in decompressed_root.iterdir() if d.is_dir()])
    print(f'✅ Số object decompressed: {len(objects)}')
    
    # Sample 1 object
    if objects:
        sample = objects[0]
        print(f'\nSample object: {sample.name}')
        sub = sorted([d for d in sample.iterdir() if d.is_dir()])
        print(f'  Số fracture: {len(sub)}')
        if sub:
            frac = sub[0]
            pieces = sorted(frac.glob('piece_*.obj'))
            print(f'  Sample fracture {frac.name}: {len(pieces)} piece(s)')
            if pieces:
                print(f'  File mẫu: {pieces[0].name} — {pieces[0].stat().st_size} bytes')
    
    print(f'\nDisk usage:')
    !du -sh {decompressed_root}

## 6. Visualize 1 mẫu — xác nhận mesh OK

In [ ]:
!pip install -q trimesh plotly

import trimesh
import numpy as np
import plotly.graph_objects as go

# Lấy 1 fracture mẫu
sample_obj = sorted(decompressed_root.iterdir())[0]
sample_frac = sorted([d for d in sample_obj.iterdir() if d.is_dir()])[0]
pieces_files = sorted(sample_frac.glob('piece_*.obj'))

print(f'Visualize object={sample_obj.name}, fracture={sample_frac.name}, {len(pieces_files)} piece(s)')

colors = ['lightblue', 'lightcoral', 'lightgreen', 'lightyellow', 'plum', 'lightsalmon', 'cyan', 'gold']

traces = []
for i, p_file in enumerate(pieces_files):
    m = trimesh.load(str(p_file))
    v = np.asarray(m.vertices)
    f = np.asarray(m.faces)
    traces.append(go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=f[:,0], j=f[:,1], k=f[:,2],
        color=colors[i % len(colors)],
        opacity=0.9, name=p_file.stem,
        flatshading=True,
    ))

fig = go.Figure(data=traces)
fig.update_layout(
    title=f'{sample_obj.name} / {sample_frac.name}',
    scene=dict(aspectmode='data'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## 7. Tạo cấu trúc output cuối + cleanup

Move `artifact/` và `data_split/` lên `/kaggle/working/` để Save Version đóng gói dễ.

In [ ]:
# Move output lên working root để Save Version
if not Path(OUTPUT_ROOT).exists():
    shutil.move(str(decompressed_root), OUTPUT_ROOT)

data_split_final = f'{WORK_DIR}/data_split'
if not Path(data_split_final).exists() and Path(f'{FILTERED_ROOT}/data_split').exists():
    shutil.move(f'{FILTERED_ROOT}/data_split', data_split_final)

# Cleanup: xóa folder filtered (đã không cần)
if Path(FILTERED_ROOT).exists():
    shutil.rmtree(FILTERED_ROOT)

# Cleanup: xóa folder repo decompress
if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

print('Cấu trúc cuối ở /kaggle/working/:')
!ls -la {WORK_DIR}/
print()
print('Tổng disk usage:')
!du -sh {WORK_DIR}/*

## 8. Tiếp theo: Save Version + Tạo Kaggle Dataset

1. Bấm **Save Version** (top right) → chọn **Save & Run All** → đợi notebook chạy lại từ đầu và snapshot output.
2. Sau khi version hoàn thành, vào **Output** tab của notebook → chọn **Create Dataset from Output** → đặt tên ví dụ `breaking-bad-artifact-decompressed`.
3. Trong notebook training tiếp theo, attach dataset này qua **Add Input**. Files sẽ ở `/kaggle/input/breaking-bad-artifact-decompressed/artifact/...`

Như vậy bạn không cần decompress lại — chỉ chạy notebook này 1 lần duy nhất.

**Output cuối có gì**:
- `artifact/` — toàn bộ mesh `piece_X.obj` đã decompress, ~3–5GB
- `data_split/` — train/val splits (`artifact.train.txt`, `artifact.val.txt`)
- `sample_manifest.json` — log fractures đã chọn (cho reproducibility)

Khi attach dataset này vào notebook training, mọi thứ sẵn sàng để viết Dataset class + train DiffusionNet.